# S11 · Ampliación: investigar 122 fuentes reales con Python

Este cuaderno complementa 01_laboratorio.ipynb. No se ejecuta un motor ajeno: escribimos bucles, mediciones, consultas, ASR y catálogos.

**Fuentes:** 30 actas SECOP con PDF + PNG de la primera página; 62 fotografías periodísticas (dos conjuntos con distinta calidad de metadatos); 30 voces humanas PolyAI MINDS-14, español, licencia CC BY 4.0. Son **122 fuentes por modalidad**, no 122 hechos relacionados.

**En el video:** procesa 5 actas, 8 fotos y 3 voces. **Para la práctica autónoma:** modifica los parámetros a 30, 62 y 30. Compara tiempos y errores, y abre cada fuente en el visor privado. No publiques datos personales de las actas.

## 1 · Construir el inventario

Aprende a recorrer carpetas con Path y a diferenciar originales de derivados. Los 30 PNG son vistas de los PDF: no cuentan como 30 actas adicionales. Calcula la huella SHA-256 y compara registros distintos por nombre con archivos idénticos por bytes.

In [ ]:
from pathlib import Path
from PIL import Image
from IPython.display import display, Audio
import pandas as pd
import hashlib, json, re, time, io, math

BASE = Path.cwd()
FOTOS = sorted((BASE/"datos/raw/fotos").glob("eltiempo_*"))
FOTOS_FICHA = sorted((BASE/"datos/raw/fotos_catalogadas").glob("*.webp"))
ACTAS = sorted((BASE/"datos/raw/actas_pdf").glob("*.pdf"))
PAGINAS = BASE/"datos/raw/actas"
OUT = BASE/"outputs"
OUT.mkdir(exist_ok=True)

def sha256(ruta):
    h = hashlib.sha256()
    with Path(ruta).open("rb") as f:
        for bloque in iter(lambda: f.read(1024*1024), b""):
            h.update(bloque)
    return h.hexdigest()

print("Fotos:", len(FOTOS), "+", len(FOTOS_FICHA))
print("Actas:", len(ACTAS), "PDF y", len(list(PAGINAS.glob("*.png"))), "PNG")
assert (len(FOTOS), len(FOTOS_FICHA), len(ACTAS)) == (30, 32, 30)
fuentes = FOTOS + FOTOS_FICHA + ACTAS
inv = pd.DataFrame([{"ruta":str(p.relative_to(BASE)), "sha256":sha256(p),
                     "mb":round(p.stat().st_size/1e6,3)} for p in fuentes])
print("Registros:",len(inv),"Huellas distintas:",inv.sha256.nunique())
display(inv.head())

## 2 · OCR documental sobre 5 o 30 actas

Tesseract reconoce caracteres de la primera página PNG; conservamos el PDF completo como documento fuente. Mide tiempo por documento y si se reconocieron palabras como TOTAL o PAGAR. **Detectar una palabra no es medir exactitud**.

En el video usa N_ACTAS=5; para la ampliación usa 30. No compartas OCR completo de documentos con identificaciones.

In [ ]:
import pytesseract

N_ACTAS = 5  # Cambiar a 30
filas_actas = []
for pdf in ACTAS[:N_ACTAS]:
    pagina = PAGINAS/(pdf.stem+"_pagina1.png")
    inicio = time.perf_counter()
    texto = pytesseract.image_to_string(Image.open(pagina),lang="spa",config="--psm 6")
    filas_actas.append({
        "pdf":pdf.name,"sha256":sha256(pdf),
        "segundos":round(time.perf_counter()-inicio,2),
        "caracteres":len(texto),"total":"TOTAL" in texto.upper(),
        "pagar":"PAGAR" in texto.upper()
    })
tabla_actas = pd.DataFrame(filas_actas)
display(tabla_actas)
print("Media de segundos:",round(tabla_actas.segundos.mean(),2))

## 3 · OCR sobre 8 o 62 fotografías

Muchas fotos tienen objetos pero no palabras. Aplica OCR y cuenta caracteres; revisa manualmente falsos positivos en el visor. Elige N_FOTOS=8 para la grabación y 62 para comparar todo el lote. La longitud de la salida NO demuestra su calidad.

In [ ]:
TODAS_FOTOS = FOTOS+FOTOS_FICHA
N_FOTOS = 8  # Cambiar a 62
filas_fotos = []
for foto in TODAS_FOTOS[:N_FOTOS]:
    inicio = time.perf_counter()
    leido = pytesseract.image_to_string(Image.open(foto),lang="spa",config="--psm 11")
    filas_fotos.append({"foto":foto.name,"caracteres":len(leido.strip()),
                         "segundos":round(time.perf_counter()-inicio,2)})
tabla_fotos = pd.DataFrame(filas_fotos)
display(tabla_fotos.sort_values("caracteres",ascending=False).head(8))

## 4 · Búsqueda visual entre 8 y 62 fotografías

Instala e inicia Ollama siguiendo INSTALACION_UBUNTU.md, y descarga embeddinggemma-2:440m, que admite imagen y texto. El modelo 270m es solo textual.

Aquí escribes la llamada HTTP, Base64, caché por SHA-256 y coseno. El primer procesamiento de 62 fotografías puede tardar en CPU. Cambia la pregunta y comprueba las primeras fotografías; la similitud no certifica la noticia.

In [ ]:
import requests, base64

MODELO = "embeddinggemma-2:440m"
API = "http://127.0.0.1:11434/api/embed"
def embedding(entrada):
    r = requests.post(API,json={"model":MODELO,"input":entrada},timeout=240)
    r.raise_for_status()
    return r.json()["embeddings"][0]

cache = OUT/"vectores_440m.json"
vectores = json.loads(cache.read_text()) if cache.exists() else {}
N_VISION = 8  # Cambiar a 62
for foto in TODAS_FOTOS[:N_VISION]:
    key = str(foto.relative_to(BASE))
    hash_actual = sha256(foto)
    if key not in vectores or vectores[key]["sha256"] != hash_actual:
        b64 = base64.b64encode(foto.read_bytes()).decode("ascii")
        vectores[key] = {"sha256":hash_actual,"vector":embedding({"image":b64})}
        cache.write_text(json.dumps(vectores),encoding="utf-8")
print("Vectores de 768 dimensiones:",len(vectores))
assert all(len(v["vector"])==768 for v in vectores.values())

In [ ]:
def coseno(a,b):
    num = sum(x*y for x,y in zip(a,b))
    norm_a = math.sqrt(sum(x*x for x in a))
    norm_b = math.sqrt(sum(y*y for y in b))
    return num/(norm_a*norm_b)

pregunta = "viviendas en una ladera"  # Cambia a medicamentos o lotería
consulta = embedding("task: search result | query: "+pregunta)
ranking = sorted(((k,coseno(consulta,v["vector"])) for k,v in vectores.items()),
                 key=lambda x:-x[1])
display(pd.DataFrame(ranking[:10],columns=["archivo","similitud"]))
display(Image.open(BASE/ranking[0][0]))

## 5 · Descargar 30 voces humanas con permiso y referencia

Fuente PolyAI MINDS-14 es-ES, licencia CC BY 4.0. Recupera el Parquet real (~39 MB) desde Hugging Face; verifica su SHA-256 antes de leer. Elige una voz de cada una de varias intenciones seleccionadas por rondas. Estos audios bancarios NO son relatos de las fotografías.

In [ ]:
from urllib.request import Request,urlopen
URL = ("https://huggingface.co/datasets/PolyAI/minds14/resolve/main/"
       "es-ES/train-00000-of-00001.parquet")
SHA_ESPERADO = "989bf2f90676a5eb5852fe785a03b1d9b97763e0febbd0fd0a3f489afd83badf"
corpus = BASE/"datos/minds14_es-ES.parquet"
if not corpus.exists():
    req=Request(URL,headers={"User-Agent":"BigData-UCentral-educativo/1.0"})
    with urlopen(req,timeout=180) as web, corpus.open("wb") as out:
        while bloque := web.read(1024*1024):
            out.write(bloque)
            if out.tell()>100_000_000: raise ValueError("Descarga excede 100 MB")
assert sha256(corpus)==SHA_ESPERADO,"Corpus distinto del auditado"
tabla = pd.read_parquet(corpus,columns=["audio","path","transcription","intent_class"])
ordenadas = tabla.sort_values("path", kind="stable").copy()
ordenadas["ronda"] = ordenadas.groupby("intent_class").cumcount()
elegidas = ordenadas.sort_values(["ronda", "intent_class", "path"], kind="stable").head(30)
assert len(elegidas)==30 and elegidas["path"].nunique()==30
print("Treinta grabaciones únicas por rondas de intención")

## 6 · Conservar los originales y convertir a WAV PCM16

Un WAV puede contener códecs distintos. SoundFile decodifica la señal y wave guarda una copia PCM16 mono compatible con el reproductor. Conserva hash de fuente y derivado. Después abre cada archivo en el visor.

In [ ]:
import soundfile as sf
import numpy as np
import wave

RAW_AUDIO=BASE/"datos/raw/audios"
DER_AUDIO=BASE/"datos/derived/audios"
RAW_AUDIO.mkdir(parents=True,exist_ok=True)
DER_AUDIO.mkdir(parents=True,exist_ok=True)
voces=[]
for n,(_,fila) in enumerate(elegidas.iterrows(),1):
    original=bytes(fila["audio"]["bytes"])
    fuente=RAW_AUDIO/f"voz_{n:02}_fuente.wav"
    if not fuente.exists(): fuente.write_bytes(original)
    assert sha256(fuente)==hashlib.sha256(original).hexdigest()
    with sf.SoundFile(io.BytesIO(original)) as dec:
        tasa=dec.samplerate
        samples=dec.read(dtype="float32",always_2d=True)
    mono=samples.mean(axis=1)
    pcm=(np.clip(mono,-1,1)*32767).astype("<i2")
    destino=DER_AUDIO/f"voz_{n:02}.wav"
    with wave.open(str(destino),"wb") as wav:
        wav.setnchannels(1);wav.setsampwidth(2);wav.setframerate(tasa)
        wav.writeframes(pcm.tobytes())
    voces.append({
        "registro":str(fila["path"]),"intencion":int(fila["intent_class"]),
        "referencia":str(fila["transcription"]),
        "fuente":str(fuente.relative_to(BASE)),"pcm16":str(destino.relative_to(BASE)),
        "sha_fuente":sha256(fuente),"sha_pcm16":sha256(destino)
    })
manifest=BASE/"datos/manifiestos/audios_minds14.json"
manifest.parent.mkdir(parents=True,exist_ok=True)
manifest.write_text(json.dumps({"dataset":"PolyAI/minds14","subset":"es-ES",
    "license":"CC BY 4.0","source_url":URL,"sha256_parquet":SHA_ESPERADO,
    "audios":voces},ensure_ascii=False,indent=2),encoding="utf-8")
display(Audio(filename=str(BASE/voces[0]["pcm16"])))
print("Audios disponibles en el visor:",len(voces))

## 7 · Medir la tasa de error de palabras con Whisper

Whisper tiny ejecuta ASR en CPU con cuantización INT8. La transcripción humana de MINDS-14 sirve como referencia del mismo registro. WER = inserciones + borrados + sustituciones / palabras de referencia. Compara errores y tiempo de tres voces en video y diez para el reto.

In [ ]:
from faster_whisper import WhisperModel

whisper=WhisperModel("tiny",device="cpu",compute_type="int8",cpu_threads=2)
def palabras(t): return re.findall(r"\w+",t.lower())
def errores(a,b):
    anterior=list(range(len(b)+1))
    for i,x in enumerate(a,1):
        fila=[i]
        for j,y in enumerate(b,1):
            fila.append(min(fila[-1]+1,anterior[j]+1,anterior[j-1]+(x!=y)))
        anterior=fila
    return anterior[-1]

N_ASR=3  # Cambiar a 30
salidas=[]
for voz in voces[:N_ASR]:
    ruta=BASE/voz["pcm16"]
    inicio=time.perf_counter()
    segmentos,_=whisper.transcribe(str(ruta),language="es",beam_size=2)
    transcripcion=" ".join(s.text.strip() for s in segmentos)
    ref=palabras(voz["referencia"])
    salidas.append({"audio":ruta.name,"wer":round(errores(ref,palabras(transcripcion))/len(ref),3),
                    "segundos":round(time.perf_counter()-inicio,2)})
tabla_wer=pd.DataFrame(salidas)
display(tabla_wer.sort_values("wer",ascending=False))

## 8 · Crear y auditar el catálogo de 122 fuentes

Hay 62 fotos, 30 actas y 30 voces, pero NO existen vínculos documentados entre estas tres colecciones. En la ETL preserva ruta, modalidad, SHA-256, crédito y la relación PDF–PNG. Los registros de las 32 fotografías con ficha tienen metadatos; para las otras 30 no inventes autor ni licencia. Evita exportar el OCR completo de actas SECOP.

In [ ]:
info=json.loads((BASE/"datos/manifiestos/fotos_catalogadas_32.json").read_text())
fichas={Path(x["input_path"]).name:x for x in info["registros"]}
catalogo=[]
for foto in TODAS_FOTOS:
    meta=fichas.get(foto.name,{})
    catalogo.append({"ruta":str(foto.relative_to(BASE)),"tipo":"foto",
        "sha256":sha256(foto),"credito":meta.get("credit") or "pendiente",
        "url":meta.get("source_url") or ""})
for pdf in ACTAS:
    pagina=PAGINAS/(pdf.stem+"_pagina1.png")
    catalogo.append({"ruta":str(pdf.relative_to(BASE)),"tipo":"acta",
        "sha256":sha256(pdf),"vista_png":str(pagina.relative_to(BASE)),"credito":"SECOP"})
for voz in voces:
    ruta=BASE/voz["fuente"]
    catalogo.append({"ruta":voz["fuente"],"tipo":"audio","sha256":sha256(ruta),
                     "vista_wav":voz["pcm16"],"credito":"PolyAI MINDS-14 CC BY 4.0"})
assert len(catalogo)==122
archivo=OUT/"catalogo_122.jsonl"
with archivo.open("w",encoding="utf-8") as f:
    for fila in catalogo: f.write(json.dumps(fila,ensure_ascii=False)+"\n")
pd.DataFrame(catalogo).to_parquet(OUT/"catalogo_122.parquet",index=False)
display(pd.DataFrame(catalogo).groupby("tipo").size())

In [ ]:
releido=[json.loads(x) for x in archivo.read_text().splitlines() if x.strip()]
parquet=pd.read_parquet(OUT/"catalogo_122.parquet")
assert len(releido)==len(parquet)==122
for fila in releido: assert sha256(BASE/fila["ruta"])==fila["sha256"]
print("OK: 122 fuentes, todos los hashes coinciden")

## Reto final de investigación

Cambiar N_ACTAS a 30, N_FOTOS a 62, N_VISION a 62 o N_ASR a 10; comparar tiempos y errores con tu muestra pequeña. Elegir un archivo en el visor y contrastarlo con el resultado. Entregar una tabla: pregunta | corpus | número de archivos | cambio en el código | resultado | evidencia | limitación.

**No es aprendizaje ejecutar todas las celdas sin entenderlas.** Demuestra cómo construiste y modificaste los bucles y cuándo elegiste OCR, búsqueda visual o ASR.

**Abrir visor privado** desde la terminal: streamlit run visor_fuentes.py --server.address 0.0.0.0 --server.port 8501